In [0]:
# Gold — Customer Dimension Analysis

from pyspark.sql.functions import regexp_extract

crm_customers = spark.table(
    "bike_lakehouse.silver.crm_customers"
)

erp_customers = spark.table(
    "bike_lakehouse.silver.erp_customers"
)

erp_locations = spark.table(
    "bike_lakehouse.silver.erp_locations"
)

df_customer_analysis = (
    crm_customers
    .join(
        erp_customers,
        crm_customers.customer_id.cast("string") ==
        regexp_extract(
            erp_customers.customer_id,
            r"(\d+)$",
            1
        ).cast("int").cast("string"),
        "left"
    )
    .join(
        erp_locations,
        crm_customers.customer_id.cast("string") ==
        regexp_extract(
            erp_locations.customer_id,
            r"(\d+)$",
            1
        ).cast("int").cast("string"),
        "left"
    )
    .select(
        crm_customers.customer_id,
        crm_customers.customer_key,
        crm_customers.first_name,
        crm_customers.last_name,
        crm_customers.marital_status,
        crm_customers.gender,
        crm_customers.create_date,
        erp_customers.birth_date,
        erp_customers.gender.alias("erp_gender"),
        erp_locations.country
    )
)

display(df_customer_analysis.limit(20))

In [0]:
# Gold — Build Customer Dimension

df_dim_customers = (
    crm_customers
    .join(
        erp_customers,
        crm_customers.customer_id.cast("string") ==
        regexp_extract(
            erp_customers.customer_id,
            r"(\d+)$",
            1
        ).cast("int").cast("string"),
        "left"
    )
    .join(
        erp_locations,
        crm_customers.customer_id.cast("string") ==
        regexp_extract(
            erp_locations.customer_id,
            r"(\d+)$",
            1
        ).cast("int").cast("string"),
        "left"
    )
    .select(
        crm_customers.customer_id,
        crm_customers.customer_key,
        crm_customers.first_name,
        crm_customers.last_name,
        crm_customers.marital_status,
        crm_customers.gender,
        crm_customers.create_date,
        erp_customers.birth_date,
        erp_locations.country
    )
)

display(df_dim_customers.limit(20))

In [0]:
# Gold — Validate Customer Dimension

print("Row count:", df_dim_customers.count())

print(
    "Duplicate customer IDs:",
    df_dim_customers.groupBy("customer_id")
    .count()
    .filter("count > 1")
    .count()
)

print(
    "NULL customer IDs:",
    df_dim_customers.filter("customer_id IS NULL").count()
)

print(
    "NULL customer keys:",
    df_dim_customers.filter("customer_key IS NULL").count()
)

print(
    "NULL birth dates:",
    df_dim_customers.filter("birth_date IS NULL").count()
)

print(
    "NULL countries:",
    df_dim_customers.filter("country IS NULL").count()
)

In [0]:
# Gold — Write Customer Dimension

df_dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.gold.dim_customers")

In [0]:
# Gold — Verify Customer Dimension

df_check = spark.table("bike_lakehouse.gold.dim_customers")

print("Gold row count:", df_check.count())

display(
    df_check.orderBy("customer_id").limit(10)
)